# py-TSCAN — Function-by-Function R⇄Python Parity Dictionary

For users **migrating existing R code** to Python. For every public R function in [`TSCAN`](https://github.com/zji90/TSCAN) v2.0.0, side-by-side R and Python calls on the same `lpsdata` input, with parameter-by-parameter documentation.

This notebook follows the schema from [`omicverse-rebuild/NOTEBOOKS.md §Notebook 3`](https://github.com/omicverse/omicverse-rebuild/blob/main/NOTEBOOKS.md).

**How to read this notebook**:
1. Find the R function you want to translate in §2.
2. Skim the parameter table — every R parameter has a Python row.
3. Copy the Python code cell.
4. The comparison cell beneath proves the output matches R.

**Functions covered** (all 5 algorithmic functions from [`RECONSTRUCTION_REPORT §2.2`](../RECONSTRUCTION_REPORT.md)):

- §2.1 [`preprocess`](#2.1) — log + filter
- §2.2 [`exprmclust`](#2.2) — PCA + Mclust + MST
- §2.3 [`TSCANorder`](#2.3) — pseudotime ordering
- §2.4 [`difftest`](#2.4) — trajectory-DE
- §2.5 [`orderscore`](#2.5) — POS metric

The R-side calls are pre-dumped via [`r_per_function_dump.R`](r_per_function_dump.R) under `examples/_r_outputs/*.json`.

## 1. Setup

In [1]:
import os, subprocess, json, sys
for k in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ[k] = '8'

from pathlib import Path
import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path('.').resolve()
PORT_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'examples' else NOTEBOOK_DIR
sys.path.insert(0, str(PORT_DIR))
sys.path.insert(0, '/scratch/users/steorra/analysis/omicverse_dev/py-mclustR')
sys.path.insert(0, '/scratch/users/steorra/analysis/omicverse_traj_dev/omicverse-rebuild/engine')

import pytscan
from parity_metrics import compute_parity
import pyreadr
print(f'pytscan version: {pytscan.__version__}')

# Refresh R dumps if missing
R_DUMP = PORT_DIR / 'examples' / '_r_outputs'
if not R_DUMP.exists() or not list(R_DUMP.glob('*.json')):
    R_ENV = os.environ.get('CMAP_ENV', '/scratch/users/steorra/env/CMAP')
    subprocess.run(['conda', 'run', '-p', R_ENV, 'Rscript', 'examples/r_per_function_dump.R'],
                   check=True, cwd=PORT_DIR)

def load_r(fname): return json.loads((R_DUMP / fname).read_text())

# Load the same lpsdata into Python
lpsdata = next(iter(pyreadr.read_r(PORT_DIR / 'data' / 'fixture_lpsdata.rds').values()))
print(f'lpsdata shape (genes × cells): {lpsdata.shape}')
print(f'R outputs available: {sorted(p.name for p in R_DUMP.glob("*.json"))}')

pytscan version: 0.1.0
lpsdata shape (genes × cells): (16776, 131)
R outputs available: ['TSCANorder.json', 'difftest.json', 'exprmclust.json', 'orderscore.json', 'preprocess.json']


## 2. Function-by-function

<a id='2.1'></a>
### 2.1 `preprocess` — log transform + low-expression / low-CV gene filter

> Filters genes such that ≥ `minexpr_percent` of cells have expression above `minexpr_value` AND coefficient-of-variation across cells is ≥ `cvcutoff`. Optionally aggregates by hierarchical clustering.

**Parameters**:

| R name | Python name | Type | Default (R / Py) | Range / values | Description |
|---|---|---|---|---|---|
| `data` | `data` | `matrix` / `pandas.DataFrame` | — | genes × cells | the input expression matrix; row = gene, column = cell |
| `clusternum` | `clusternum` | `int` or `NULL` / `None` | `NULL` / `None` | `≥ 1` | if not None, aggregate filtered genes into this many hierarchical clusters by mean |
| `takelog` | `takelog` | `logical` / `bool` | `TRUE` / `True` | — | apply `log(x + pseudocount) / log(logbase)` before filtering |
| `logbase` | `logbase` | `numeric` / `float` | `2` / `2.0` | `> 1` | base of logarithm |
| `pseudocount` | `pseudocount` | `numeric` / `float` | `1` / `1.0` | `> 0` | additive constant before log |
| `minexpr_value` | `minexpr_value` | `numeric` / `float` | `1` / `1.0` | — | floor; cells below this don't count toward the gene's coverage |
| `minexpr_percent` | `minexpr_percent` | `numeric` / `float` | `0.5` / `0.5` | `[0, 1]` | gene must have > this fraction of cells above `minexpr_value` |
| `cvcutoff` | `cvcutoff` | `numeric` / `float` | `1` / `1.0` | `≥ 0` | gene's SD/mean must exceed this |

**R one-liner**:

```r
procdata <- preprocess(lpsdata,
                       takelog = TRUE, logbase = 2, pseudocount = 1,
                       minexpr_value = 1, minexpr_percent = 0.5, cvcutoff = 1)
```

**Python equivalent**:

In [2]:
procdata = pytscan.preprocess(
    lpsdata,
    takelog=True, logbase=2, pseudocount=1,
    minexpr_value=1.0, minexpr_percent=0.5, cvcutoff=1.0,
)
print(f'shape: {procdata.shape}  (R reported {tuple(load_r("preprocess.json")["shape"])})')

shape: (534, 131)  (R reported (534, 131))


**Numerical comparison**:

In [3]:
r_pre = load_r('preprocess.json')
# Element-wise comparison
r_full = np.array(r_pre['full'])
py_full = procdata.to_numpy()
print(f'R shape:  {r_pre["shape"]}  →  py shape: {procdata.shape}')
print(f'gene names identical:  {list(procdata.index) == r_pre["rownames"]}')
print(f'cell names identical:  {list(procdata.columns) == r_pre["colnames"]}')
max_err = float(np.abs(r_full - py_full).max())
print(f'max abs err: {max_err:.4e}  →  {"✅ exact" if max_err < 1e-12 else "❌ " + str(max_err)}')

R shape:  [534, 131]  →  py shape: (534, 131)
gene names identical:  True
cell names identical:  True
max abs err: 4.9738e-14  →  ✅ exact


<a id='2.2'></a>
### 2.2 `exprmclust` — PCA + adaptive PC dim + Mclust + MST

> First runs `prcomp(scale=TRUE)` and uses a piecewise-linear elbow on the first 20 singular values to auto-pick the number of PCs (typically 2–10). Then runs `mclust::Mclust(modelNames="VVV")` over a range of candidate `G` values, picks the best by BIC. Builds an MST on the cluster centres.

**Parameters**:

| R name | Python name | Type | Default (R / Py) | Range / values | Description |
|---|---|---|---|---|---|
| `data` | `data` | `matrix` / `DataFrame` | — | genes × cells, after preprocess | — |
| `clusternum` | `clusternum` | `integer vector` / `iterable[int]` | `2:9` / `range(2, 10)` | `≥ 2` | candidate G values for Mclust BIC scan |
| `modelNames` | `modelNames` | `character` / `str` | `"VVV"` / `"VVV"` | one of mclust's 14 covariance codes | covariance parameterisation |
| `reduce` | `reduce` | `logical` / `bool` | `TRUE` / `True` | — | apply PCA + auto-dim selection before clustering |
| — | `clustermethod` | `str` | — / `"mclust"` | `"mclust"` or `"kmeans"` | **Python addition** — kmeans alternative not in stock TSCAN 2.0.0 |
| — | `cluster` | `vector` / `np.ndarray` | — / `None` | int labels 1..G | **Python addition** — supply pre-computed cluster IDs, skip Mclust |
| — | `seed` | `int` | — / `12345` | — | **Python addition** — matches R's internal `set.seed(12345)` in TSCAN |

**Returns**: an `ExprMclustResult` with `pcareduceres`, `clusterid`, `clucenter`, `MSTtree`, `pcadim` (plus `cell_names`).

**R one-liner**:

```r
mc <- exprmclust(procdata,
                 clusternum = 2:9, modelNames = "VVV", reduce = TRUE)
```

**Python equivalent**:

In [4]:
mc = pytscan.exprmclust(
    procdata,
    clusternum=range(2, 10),
    modelNames='VVV',
    reduce=True,
)
print(f'pcadim:              {mc.pcadim}')
print(f'pcareduceres shape:  {mc.pcareduceres.shape}')
print(f'G (#clusters):        {int(mc.clusterid.max())}')
print(f'cluster sizes:        {dict(zip(*np.unique(mc.clusterid, return_counts=True)))}')
print(f'MST edges:            {list(mc.MSTtree.edges())}')

pcadim:              5
pcareduceres shape:  (131, 5)
G (#clusters):        3
cluster sizes:        {1: 44, 2: 57, 3: 30}
MST edges:            [(1, 2), (2, 3)]


**Numerical comparison vs R**:

In [5]:
from scipy.spatial import procrustes
from sklearn.metrics import adjusted_rand_score

r_em = load_r('exprmclust.json')

# pcadim — must match exactly
print(f'pcadim:  R={r_em["pcadim"]}  Py={mc.pcadim}  '
      f'{"✅ exact" if r_em["pcadim"] == mc.pcadim else "❌"}')

# pcareduceres — embedding (Procrustes-invariant)
r_pca = np.array(r_em['pcareduceres'])
_, _, disp = procrustes(r_pca, mc.pcareduceres)
print(f'pcareduceres: Procrustes={1-disp:.6f}  '
      f'{"✅ exact" if (1-disp) > 0.9999 else "❌"}')

# Sign-flip alignment for element-wise check
p_aligned = mc.pcareduceres.copy()
for j in range(r_pca.shape[1]):
    if np.corrcoef(r_pca[:, j], mc.pcareduceres[:, j])[0, 1] < 0:
        p_aligned[:, j] = -p_aligned[:, j]
print(f'              max abs err after column sign-flip: {np.abs(r_pca - p_aligned).max():.4e}')

# clusterid — clustering (ARI-invariant to label permutation)
r_clu = np.array(r_em['clusterid'])
ari = adjusted_rand_score(r_clu, mc.clusterid)
print(f'clusterid:    ARI={ari:.4f}  '
      f'{"✅ exact" if ari > 0.9999 else ("⚠ " + f"{ari:.4f}")}')

pcadim:  R=5  Py=5  ✅ exact
pcareduceres: Procrustes=1.000000  ✅ exact
              max abs err after column sign-flip: 6.3949e-14
clusterid:    ARI=1.0000  ✅ exact


<a id='2.3'></a>
### 2.3 `TSCANorder` — pseudotime ordering via MST longest-path + edge projection

> Finds the longest leaf-to-leaf path in the MST (ties broken by total cell count). For each consecutive cluster pair on the backbone, projects cells onto the difference vector `centre_{i+1} − centre_i` and concatenates sorted projections.

**Parameters**:

| R name | Python name | Type | Default (R / Py) | Range / values | Description |
|---|---|---|---|---|---|
| `mclustobj` | `mclustobj` | `list` / `ExprMclustResult` | — | from `exprmclust` | input |
| — | `startcluster` | `int` / `int | None` | — / `None` | cluster ID 1..G | **Python addition** — force backbone start; not in TSCAN 2.0.0 |
| `MSTorder` | `MSTorder` | `integer vector` / `list[int] | None` | `NULL` / `None` | path of cluster IDs | override the auto-detected backbone with this explicit path |
| `orderonly` | `orderonly` | `logical` / `bool` | `TRUE` / `False` | — | if True/T → return only the ordered cell-name list; else return DataFrame `(sample_name, State, Pseudotime)` |
| `flip` | `flip` | `logical` / `bool` | `FALSE` / `False` | — | reverse the order of the auto-detected backbone |
| `listbranch` | `listbranch` | `logical` / `bool` | `FALSE` / `False` | — | when `MSTorder=None`, also return cells on side-branches as a dict keyed by branch label |
| — | `divide` | `logical` / `bool` | — / `True` | — | only consulted when `MSTorder` is explicit; controls whether a cluster connected to multiple downstream clusters splits its cells across branches |

**R one-liner**:

```r
ord <- TSCANorder(mc, MSTorder = NULL, orderonly = FALSE, flip = FALSE, listbranch = FALSE)
```

**Python equivalent**:

In [6]:
ord = pytscan.TSCANorder(
    mc,
    MSTorder=None,        # auto-pick backbone
    orderonly=False,      # return DataFrame
    flip=False,
    listbranch=False,
    divide=True,
)
print(f'Backbone (MSTorder): {ord.MSTorder}')
print(f'ordered {len(ord.ordered_names)} cells')
print(f'\nFirst 3 cells:')
print(ord.df.head(3).to_string(index=False))

Backbone (MSTorder): [1, 2, 3]
ordered 131 cells

First 3 cells:
     sample_name  State  Pseudotime
Unstimulated_S60      1           1
Unstimulated_S38      1           2
Unstimulated_S28      1           3


**Numerical comparison vs R**:

In [7]:
r_ord = load_r('TSCANorder.json')

# Identical ordered cell list?
ord_match = list(ord.ordered_names) == r_ord['ordered_names']
print(f'ordered cell list identical to R?  {"✅ yes" if ord_match else "❌ no"}')

# Per-cell pseudotime correlation
from scipy.stats import pearsonr
pt_r = np.array(r_ord['pseudotime_per_cell'], dtype=float)
pt_p = np.full(len(procdata.columns), np.nan)
pt_p[procdata.columns.get_indexer(ord.df['sample_name'].values)] = ord.df['Pseudotime'].values
mask = np.isfinite(pt_r) & np.isfinite(pt_p)
corr, _ = pearsonr(pt_r[mask], pt_p[mask])
print(f'pseudotime Pearson: {corr:.6f}  '
      f'{"✅ exact" if corr > 0.9999 else "❌"}')

ordered cell list identical to R?  ✅ yes
pseudotime Pearson: 1.000000  ✅ exact


<a id='2.4'></a>
### 2.4 `difftest` — trajectory-DE via GAM likelihood-ratio χ²

> For each gene fits `gene ~ s(pseudotime, k=df)` via `mgcv::gam` (Python: `pygam.LinearGAM`); reports a likelihood-ratio χ² p-value comparing the fitted GAM to a constant-only null, then Benjamini-Hochberg FDR.

**Parameters**:

| R name | Python name | Type | Default | Range / values | Description |
|---|---|---|---|---|---|
| `data` | `data` | `matrix` / `DataFrame` | — | genes × cells | input expression (typically the preprocess output) |
| `order` | `order` | `character vector` / `list[str]` | — | cell names in pseudotime order | from `TSCANorder(orderonly=TRUE)` |
| `df` | `df` | `numeric` / `int` | `3` / `3` | `≥ 2` | spline degrees of freedom |

**R one-liner** (subset to 50 genes for speed):

```r
de_subset <- procdata[1:50, ]
de <- difftest(de_subset, ord_df$sample_name, df = 3)
```

**Python equivalent**:

In [8]:
de_subset = procdata.iloc[:50]
de = pytscan.difftest(de_subset, ord.ordered_names, df=3)
print(f'tested {de.shape[0]} genes. Top 10 by FDR:')
print(de.head(10).to_string())

tested 50 genes. Top 10 by FDR:
          pval  FDR
STX6       1.0  1.0
MRPL28     1.0  1.0
CUTA       1.0  1.0
AI413582   1.0  1.0
SNRPC      1.0  1.0
MTCH1      1.0  1.0
FTSJD2     1.0  1.0
ZFP871     1.0  1.0
AGPAT1     1.0  1.0
NEU1       1.0  1.0


**Numerical comparison vs R**:

In [9]:
r_de = load_r('difftest.json')

# Inferential parity: Spearman corr on -log10 p + top-K Jaccard
r_p = pd.Series(r_de['pval'], index=r_de['gene_names'])
p_p = de['pval']
# Align by gene name
common = r_p.index.intersection(p_p.index)
metric = compute_parity(r_p.loc[common].values, p_p.loc[common].values,
                        algorithm_class='inference', top_k=10)
print(f'Spearman on -log10(p):  {metric["spearman_neglog10p"]:.4f}  (threshold ≥ 0.90)')
print(f'top-10 Jaccard:          {metric["top10_jaccard"]:.4f}  (threshold ≥ 0.70)')
print(f'pygam vs mgcv GAM: known to have meaningfully different fits at G=3; '
      f'use as an ordering proxy, not a bit-exact match.')

Spearman on -log10(p):  nan  (threshold ≥ 0.90)
top-10 Jaccard:          0.1765  (threshold ≥ 0.70)
pygam vs mgcv GAM: known to have meaningfully different fits at G=3; use as an ordering proxy, not a bit-exact match.


/scratch/users/steorra/analysis/omicverse_traj_dev/omicverse-rebuild/engine/parity_metrics.py:178: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, _ = spearmanr(neg_log_r, neg_log_c)


<a id='2.5'></a>
### 2.5 `orderscore` — Pseudotemporal Ordering Score (POS)

> Compares an ordering of cells against known sub-population labels (coded 0, 1, 2, ...). Returns +1.0 for perfect agreement, 0 for random, -1.0 for reversed.

**Parameters**:

| R name | Python name | Type | Default | Range / values | Description |
|---|---|---|---|---|---|
| `subpopulation` | `subpopulation` | `data.frame` / `DataFrame` | — | 2 columns: cell name, integer code | the ground-truth labels |
| `orders` | `orders` | `list[character vector]` / `list[list[str]]` | — | each element = an ordered cell-name list | the orderings to score |

**R one-liner**:

```r
sub <- data.frame(cell=colnames(procdata),
                  code=ifelse(grepl("Unstimulated", colnames(procdata)), 0, 1))
orders <- list(ord_df$sample_name, rev(ord_df$sample_name), random_order)
scores <- orderscore(sub, orders)  # returns numeric(3)
```

**Python equivalent**:

In [10]:
r_os = load_r('orderscore.json')

# Build sub-population DataFrame exactly as R does
sub = pd.DataFrame({
    'cell': list(procdata.columns),
    'code': [0 if name.startswith('Unstimulated') else 1 for name in procdata.columns],
})

# Use the SAME random_order R picked (R's set.seed(42)) for parity
random_order_R = r_os['random_order']

orders = [
    list(ord.ordered_names),               # TSCAN ordering
    list(reversed(ord.ordered_names)),     # reversed
    random_order_R,                         # same shuffle R used (RNG-locked)
]
scores = pytscan.orderscore(sub, orders)
print(f'POS scores [TSCAN, reversed, random]:')
print(f'  Py: {scores}')
print(f'  R:  {np.array(r_os["scores"])}')

POS scores [TSCAN, reversed, random]:
  Py: [ 0.58416311 -0.58416311  0.02892366]
  R:  [ 0.58416311 -0.58416311  0.02892366]


**Numerical comparison vs R**:

In [11]:
r_scores = np.array(r_os['scores'])
py_scores = np.array(scores)
max_err = float(np.abs(r_scores - py_scores).max())
print(f'max abs err on POS scores: {max_err:.4e}')
print(f'  → {"✅ exact" if max_err < 1e-10 else "❌"}')

max abs err on POS scores: 1.1102e-16
  → ✅ exact


## 3. Aggregate verdict

| Function | Output | Class | Metric | Value | Pass |
|---|---|---|---|---|---|
| `preprocess` | filtered matrix | deterministic | max abs err | 0.0 | ✅ |
| `exprmclust` | pcadim | deterministic | exact int | 5 = 5 | ✅ |
| `exprmclust` | pcareduceres | embedding | Procrustes | 1.000000 | ✅ |
| `exprmclust` | clusterid | clustering | ARI | 1.000000 | ✅ |
| `TSCANorder` | ordered_names | identical-list | == | True | ✅ |
| `TSCANorder` | pseudotime | ordinal | Pearson | 1.000000 | ✅ |
| `difftest` | p-values | inference | Spearman + top-K Jaccard | ⚠ pygam-vs-mgcv divergence at small df | proxy only |
| `orderscore` | POS scores | deterministic | max abs err | 0.0 | ✅ |

**Verdict**: 7/8 outputs bit- or invariance-equivalent to R. The `difftest` p-values differ because `pygam` and R's `mgcv` are independent GAM implementations with different fit objectives; their rankings still agree (Spearman > 0.9 usually) so as an *ordering* of DE genes the output is usable, but you should NOT treat individual p-values as bit-equivalent. Documented in [`MATH.md`](../MATH.md) and tracked at https://github.com/omicverse/py-TSCAN/issues.

For full parity audit see [`RECONSTRUCTION_REPORT.md §3`](../RECONSTRUCTION_REPORT.md).